# Batch docking the whole library into COX-2 (AutoDock Vina)
### Dock all 24 ferulic derivatives, rank them, find where FA19 lands

gnina was giving you trouble (the CNN binary needs a specific GPU/CUDA setup). **AutoDock Vina is the reliable open-source choice** — pure CPU, no fragile binary — so this docks your **entire `library.csv`** into COX-2 in one pass and ranks every compound by predicted binding. This is the real screening funnel step: properties already triaged the library; now docking scores binding.

**What it does:** prepare COX-2 once → loop over all 24 ligands (SMILES → 3D → dock → record best affinity) → rank → plot → validate with a celecoxib re-dock.

> Runs on a normal Colab CPU runtime. ~1–3 min per molecule at `exhaustiveness=8`, so budget ~30–60 min for 24. Lower exhaustiveness = faster/noisier.


## 1 · Install
*Why each:* `vina` = docking engine; `meeko` + `gemmi` = ligand → PDBQT; `rdkit` = 3D structures; `openbabel` = receptor prep + a ligand fallback.

In [ ]:
!pip -q install vina meeko gemmi rdkit requests pandas matplotlib 2>/dev/null
!apt-get -qq install -y openbabel >/dev/null 2>&1
import os, requests, numpy as np, pandas as pd
from rdkit import Chem
from rdkit.Chem import AllChem
from rdkit import RDLogger; RDLogger.DisableLog('rdApp.*')
from vina import Vina
print("ready")

## 2 · Prepare the receptor **once**
Download COX-2 (3LN1), keep chain A protein, drop water/ligand/cofactors, protonate, write PDBQT. Also read the bound celecoxib to place the search box objectively at the known site.


In [ ]:
pdb = requests.get("https://files.rcsb.org/download/3LN1.pdb", timeout=60).text
open("3LN1.pdb","w").write(pdb)
with open("receptor_clean.pdb","w") as f:
    for l in pdb.splitlines():
        if l.startswith("ATOM") and l[21]=="A": f.write(l+"\n")
    f.write("TER\nEND\n")
!obabel receptor_clean.pdb -O receptor.pdbqt -xr -p 7.4 2>/dev/null

cel = [[float(l[30:38]),float(l[38:46]),float(l[46:54])]
       for l in pdb.splitlines()
       if l.startswith("HETATM") and l[17:20].strip()=="CEL" and l[21]=="A"]
CENTER = list(np.mean(cel, axis=0).round(2)); BOX = [24.0,24.0,24.0]
print("receptor ready | box center", CENTER, "size", BOX)

## 3 · Load the library
All 24 molecules. (Upload `library.csv` to the Colab session first, or mount Drive.)


In [ ]:
lib = pd.read_csv("library.csv")          # columns: id, klass, smiles
print(f"{len(lib)} molecules to dock"); lib.head()

## 4 · The per-molecule prep + dock function
For each SMILES: build a 3D conformer (RDKit), write PDBQT (Meeko, with an Open Babel fallback so one awkward molecule can't break the run), then dock in the fixed box and take the **best** pose's affinity. Everything is wrapped in `try/except` so a single failure is logged, not fatal.


In [ ]:
def prep_ligand(smi, path, seed=42):
    m = Chem.MolFromSmiles(smi)
    if m is None: return False
    m = Chem.AddHs(m)
    if AllChem.EmbedMolecule(m, randomSeed=seed) != 0: return False
    AllChem.MMFFOptimizeMolecule(m)
    try:
        from meeko import MoleculePreparation, PDBQTWriterLegacy
        setups = MoleculePreparation().prepare(m)
        open(path,"w").write(PDBQTWriterLegacy.write_string(setups[0])[0])
    except Exception:                              # fallback: Open Babel
        Chem.MolToMolFile(m, "tmp.mol")
        os.system(f"obabel tmp.mol -O {path} -p 7.4 2>/dev/null")
    return os.path.exists(path) and os.path.getsize(path) > 0

def dock(path, exhaustiveness=8):
    v = Vina(sf_name='vina', cpu=0, seed=42, verbosity=0)
    v.set_receptor("receptor.pdbqt")
    v.set_ligand_from_file(path)
    v.compute_vina_maps(center=CENTER, box_size=BOX)
    v.dock(exhaustiveness=exhaustiveness, n_poses=5)
    return float(v.energies(n_poses=1)[0][0])      # best-pose affinity (kcal/mol)

## 5 · Dock the whole library
We also compute **ligand efficiency** = affinity ÷ heavy atoms, which fairly compares big and small molecules (a large molecule binding hard isn't impressive if it's just big).

In [ ]:
rows=[]
for i, r in lib.iterrows():
    try:
        ok = prep_ligand(r["smiles"], "lig.pdbqt")
        if not ok: raise RuntimeError("prep failed")
        aff = dock("lig.pdbqt")
        n_heavy = Chem.MolFromSmiles(r["smiles"]).GetNumHeavyAtoms()
        rows.append({"id":r["id"], "klass":r.get("klass",""), "affinity":round(aff,2),
                     "lig_eff":round(aff/n_heavy,3)})
        print(f"  [{i+1:>2}/{len(lib)}] {r['id']:5s} {aff:7.2f} kcal/mol")
    except Exception as e:
        rows.append({"id":r["id"], "klass":r.get("klass",""), "affinity":None, "lig_eff":None})
        print(f"  [{i+1:>2}/{len(lib)}] {r['id']:5s} FAILED: {e}")

res = pd.DataFrame(rows).sort_values("affinity")     # most negative = best
res.insert(0, "rank", range(1, len(res)+1))
res.to_csv("docking_results.csv", index=False)
print("\n=== ranked by Vina affinity (top = best) ===")
print(res.to_string(index=False))

## 6 · Plot the ranking (FA19 highlighted)

In [ ]:
import matplotlib.pyplot as plt
d = res.dropna(subset=["affinity"]).sort_values("affinity")
colors = ["#c44536" if i=="FA19" else "#2a9d8f" for i in d["id"]]
plt.figure(figsize=(10,6))
plt.barh(d["id"], d["affinity"], color=colors)
plt.gca().invert_yaxis()
plt.xlabel("Vina best-pose affinity (kcal/mol — more negative = stronger)")
plt.title("Whole-library docking into COX-2 (FA19 in red)")
plt.tight_layout(); plt.savefig("docking_ranking.png", dpi=150); plt.show()
print("FA19 rank:", int(res[res['id']=='FA19']['rank'].iloc[0]), "of", res['affinity'].notna().sum())

## 7 · Validate the protocol — re-dock celecoxib
The whole ranking is only trustworthy if the setup reproduces a known pose. Re-dock the crystal celecoxib; recovering it within **~2 Å RMSD** is the green light.


In [ ]:
with open("cel.pdb","w") as f:
    for l in pdb.splitlines():
        if l.startswith("HETATM") and l[17:20].strip()=="CEL" and l[21]=="A": f.write(l+"\n")
    f.write("END\n")
!obabel cel.pdb -O cel.pdbqt -p 7.4 2>/dev/null
!obabel cel.pdb -O cel_crystal.sdf 2>/dev/null
vc = Vina(sf_name='vina', cpu=0, seed=42, verbosity=0)
vc.set_receptor("receptor.pdbqt"); vc.set_ligand_from_file("cel.pdbqt")
vc.compute_vina_maps(center=CENTER, box_size=BOX); vc.dock(exhaustiveness=8, n_poses=1)
vc.write_poses("cel_redock.pdbqt", n_poses=1, overwrite=True)
!obabel cel_redock.pdbqt -O cel_redock.sdf -f 1 -l 1 2>/dev/null
print("celecoxib redock affinity:", round(float(vc.energies(n_poses=1)[0][0]),2), "kcal/mol")
print("RMSD vs crystal (want < 2.0):")
!obrms cel_crystal.sdf cel_redock.sdf

## 8 · Combine with the screening (the funnel)
If you have `screening_results.csv` from the property pipeline, merge it so each compound has **both** drug-likeness and docking — that's the complete picture: clean *and* binds.


In [ ]:
try:
    scr = pd.read_csv("screening_results.csv")[["id","QED","SAscore","Lipinski_violations","pass_prefilter"]]
    funnel = res.merge(scr, on="id", how="left").sort_values("affinity")
    funnel.to_csv("funnel_dock_plus_properties.csv", index=False)
    print(funnel.to_string(index=False))
except FileNotFoundError:
    print("screening_results.csv not found — skip merge (docking_results.csv is still saved)")

## Notes
- **Why Vina, not gnina:** Vina is CPU-only with no special binary, so it just runs. gnina's CNN adds a learned score but needs a working GPU/CUDA build — use it later if you want the AI scoring, but Vina is the dependable workhorse for a full library.
- **Speed vs. quality:** raise `exhaustiveness` (16–32) for a more careful final ranking of your top hits; keep it at 8 for a fast first pass over all 24.
- **Interpretation:** Vina won't equal your MOE numbers (different scoring function); what matters is the *ranking* and whether FA19 sits among the stronger binders. Always read it together with the re-dock RMSD control.
- **Outputs saved:** `docking_results.csv`, `docking_ranking.png`, and (if merged) `funnel_dock_plus_properties.csv`.
